# 15. Embedding Analysis（spec §10）

## 学習目標
- token embedding のノルム・学習前後のドリフト・近傍・2次元投影を観察する
- 2次元投影の解釈上の危険性（情報損失・手法依存）を理解する

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
A = M2_RUN / "analysis"
emb = load_json(A / "embedding_stats.json")
pca = np.load(A / "embedding_pca.npz")
print(f"embedding norm: mean {emb['norm_mean']:.2f} ± {emb['norm_std']:.2f}")
print(f"init→final ドリフト: mean {emb['drift_mean']:.3f}, max {emb['drift_max']:.3f}")

# ノルム vs ドリフトの散布（頻出/稀トークンで挙動が違う）
fig = go.Figure(go.Scatter(x=pca["norms"], y=pca["drift"], mode="markers",
                           marker=dict(size=4, opacity=0.5, color=pca["drift"], colorscale="Viridis"),
                           text=[f"id {i}" for i in pca["ids"]]))
fig.update_layout(title="token embedding: ノルム vs 学習ドリフト",
                  xaxis_title="‖embedding‖ (final)", yaxis_title="‖final − init‖",
                  template="plotly_white", height=380)
fig.show()

embedding norm: mean 0.59 ± 0.19
init→final ドリフト: mean 0.450, max 0.929


In [3]:
# PCA 2次元投影
fig = go.Figure(go.Scatter(x=pca["coords"][:,0], y=pca["coords"][:,1], mode="markers",
                           marker=dict(size=4, opacity=0.5, color=pca["norms"], colorscale="Plasma",
                                       colorbar_title="norm"),
                           text=[f"id {i}" for i in pca["ids"]]))
fig.update_layout(title="token embedding の PCA 2次元投影（色=ノルム）",
                  xaxis_title="PC1", yaxis_title="PC2", template="plotly_white", height=420)
fig.show()

In [4]:
# 近傍トークン（学習済みモデルを読み込んで）
from jp_llm_lab.reporting.analysis_artifacts import _load_model, _load_tokenizer
from jp_llm_lab.instrumentation import embedding_analysis as ea
model = _load_model(sorted(M2_RUN.glob("checkpoints/*100pct*.pt"))[0])
tok = _load_tokenizer(M2_RUN)
for word in ["東京", "です", "年"]:
    tid = tok.encode(word)
    if tid:
        nbrs = ea.nearest_neighbors(model, tid[0], k=6)
        near = [tok.id_to_token(i) for i,_ in nbrs]
        print(f"{word!r}(id {tid[0]}) の近傍: {near}")

'東京'(id 6479) の近傍: ['■', '通常', '2014年', 'Web', '上記', '標準']
'です'(id 5809) の近傍: ['です。', 'でしょう', 'でした', 'と言', 'と思', 'だった']
'年'(id 2043) の近傍: ['日', '月', '回', '年の', '時', 'つ']


## Observation / Interpretation / Caveat
- **Observation**: 埋め込みノルムには幅があり、ドリフトはトークンにより大きく異なる（頻出トークンほど大きく動く傾向）。PCA 投影に緩いクラスタ構造が見える。
- **Interpretation**: よく出るトークンは勾配を多く受けるため init から大きく動く。近傍は表層的な共起を反映しがちで、意味的近さとは限らない。
- **Caveat**: PCA 2次元は高次元情報を大きく捨てる。クラスタの見え方は手法（PCA/UMAP/t-SNE）に依存し、距離を過度に解釈してはいけない。稀トークンの近傍は init 由来のノイズを含む。

**次へ**: [16_residual_stream_analysis](16_residual_stream_analysis.ipynb)